# 18. 텍스트 단어 연관 규칙 분석 (Association Rule Text)

`Pinocchio.txt` 대사 텍스트를 전처리하고 `TransactionEncoder`와 `apriori`, `association_rules`를 적용해 함께 등장하는 단어 패턴 및 바이그램(Bigram)을 시각화합니다.

In [ ]:
import re
import warnings
from pathlib import Path
from collections import Counter
import pandas as pd
import matplotlib.pyplot as plt
import networkx as nx
from sklearn.feature_extraction.text import ENGLISH_STOP_WORDS
from mlxtend.preprocessing import TransactionEncoder
from mlxtend.frequent_patterns import apriori, association_rules

warnings.filterwarnings('ignore')

candidates = [
    Path('Study_Python/[7] Machine_Learning/data/Pinocchio.txt'),
    Path('../[7] Machine_Learning/data/Pinocchio.txt'),
    Path('data/Pinocchio.txt'),
]
txt_path = next((p for p in candidates if p.is_file()), None)
if txt_path is None:
    raise FileNotFoundError('Pinocchio.txt 파일을 찾을 수 없습니다.')

try:
    lines = txt_path.read_text(encoding='utf-8').splitlines()
except UnicodeDecodeError:
    lines = txt_path.read_text(encoding='latin-1').splitlines()

print(f'총 줄 수: {len(lines)}')

In [ ]:
# 불용어 정의 및 줄별 단어 트랜잭션 생성
stop_words = set(ENGLISH_STOP_WORDS)
stop_words.update(['oh', 'ah', 'uh', 'huh', 'hey', 'ha', 'ho', 'let', 'got', 'know', 'come', 'going', 'say', 'said'])

transactions = []
for line in lines:
    words = re.findall(r"[a-z]+(?:'[a-z]+)?", line.lower())
    filtered = sorted(set(w for w in words if len(w) > 2 and w not in stop_words and "'" not in w))
    if len(filtered) >= 2:
        transactions.append(filtered)

encoder = TransactionEncoder()
encoded = encoder.fit_transform(transactions)
df_trans = pd.DataFrame(encoded, columns=encoder.columns_)
print('트랜잭션 행렬 크기:', df_trans.shape)
print(df_trans.iloc[:5, :8])

In [ ]:
# Apriori 빈발 단어 집합 및 연관 규칙 생성
min_count = 5
frequent_items = apriori(df_trans, min_support=min_count / len(df_trans), use_colnames=True, max_len=2)
rules = association_rules(frequent_items, metric='confidence', min_threshold=0.2)
rules = rules[rules['lift'] > 1.2].copy()
rules['from'] = rules['antecedents'].apply(lambda x: ', '.join(sorted(x)))
rules['to'] = rules['consequents'].apply(lambda x: ', '.join(sorted(x)))
rules = rules.sort_values('lift', ascending=False)
print(rules[['from', 'to', 'support', 'confidence', 'lift']].head(10))

In [ ]:
# 네트워크 그래프 & 상위 10개 규칙 Lift 막대그래프 시각화
top_rules = rules.head(12)
G = nx.DiGraph()
for _, row in top_rules.iterrows():
    G.add_edge(row['from'], row['to'], confidence=row['confidence'])

plt.figure(figsize=(10, 6))
pos = nx.spring_layout(G, seed=42)
nx.draw(G, pos, with_labels=True, node_color='lightblue', node_size=1800, font_size=9, arrows=True)
plt.title('Word Association Network (Top 12)')
plt.show()

plot_data = rules.head(10).sort_values('lift')
labels = plot_data['from'] + ' -> ' + plot_data['to']
plt.figure(figsize=(9, 5))
plt.barh(labels, plot_data['lift'], color='steelblue')
plt.xlabel('Lift')
plt.title('Top 10 Word Association Rules')
plt.tight_layout()
plt.show()

In [ ]:
# 인접 두 단어(Bigram) 빈도 분석
bigram_counts = Counter()
for line in lines:
    parts = re.split(r'[.!?,;:]+', line.lower())
    for part in parts:
        words = [w for w in re.findall(r"[a-z]+", part) if len(w) > 2 and w not in stop_words]
        for i in range(len(words) - 1):
            bigram_counts[(words[i], words[i + 1])] += 1

print('상위 10개 Bigram:', bigram_counts.most_common(10))